<a href="https://colab.research.google.com/github/tasyriqomar/ColabMD-Edu_Protein-NA/blob/main/Training_Set_Docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Mount to GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# @title Create the folder > Training Set Docking
!mkdir -p "/content/drive/MyDrive/Training_Set_Docking"

In [3]:
# @title Change Directory > Training Set Docking
import os
os.chdir('/content/drive/MyDrive/Training_Set_Docking')

In [4]:
# @title 1. Install Dependencies { display-mode: "form" }
!pip install biopython scipy requests
!pip install torch_geometric
!apt-get update
!apt-get install dssp cd-hit -y
print("Dependencies installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.6 MB/s  0:00:00
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,132 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [82.5 kB]
Get:8 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backpo

In [5]:
# @title 2. Download Protein–Nucleic Acid Complexes (Fast Multithreaded) { display-mode: "form" }
import os
import gzip
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_protein_na_pdb_ids():
    search_url = "https://search.rcsb.org/rcsbsearch/v2/query"
    pdb_ids = []
    start = 0
    rows = 5000

    print("Fetching Protein–Nucleic Acid PDB IDs from RCSB API...")
    while True:
        query = {
            "query": {
                "type": "group",
                "logical_operator": "and",
                "nodes": [
                    {
                        "type": "terminal",
                        "service": "text",
                        "parameters": {
                            "attribute": "rcsb_entry_info.polymer_entity_count_protein",
                            "operator": "greater_or_equal",
                            "value": 1
                        }
                    },
                    {
                        "type": "group",
                        "logical_operator": "or",
                        "nodes": [
                            {
                                "type": "terminal",
                                "service": "text",
                                "parameters": {
                                    "attribute": "rcsb_entry_info.polymer_entity_count_DNA",
                                    "operator": "greater_or_equal",
                                    "value": 1
                                }
                            },
                            {
                                "type": "terminal",
                                "service": "text",
                                "parameters": {
                                    "attribute": "rcsb_entry_info.polymer_entity_count_RNA",
                                    "operator": "greater_or_equal",
                                    "value": 1
                                }
                            },
                            {
                                "type": "terminal",
                                "service": "text",
                                "parameters": {
                                    "attribute": "rcsb_entry_info.polymer_entity_count_nucleic_acid_hybrid",
                                    "operator": "greater_or_equal",
                                    "value": 1
                                }
                            }
                        ]
                    }
                ]
            },
            "return_type": "entry",
            "request_options": {
                "results_content_type": ["experimental"],
                "paginate": {"start": start, "rows": rows}
            }
        }

        response = requests.post(search_url, json=query, timeout=30)
        if response.status_code == 204:
            break
        if response.status_code != 200:
            print(f"API Error or finished: {response.status_code}")
            break

        data = response.json()
        result_set = data.get("result_set", [])
        if not result_set:
            break

        pdb_ids.extend(r["identifier"] for r in result_set)
        print(f"Found {len(pdb_ids)} protein–nucleic acid complexes so far...")

        if len(result_set) < rows:
            break

        start += rows
        time.sleep(0.2)

    return list(set(pdb_ids))

def _download_one(pdb_id, output_dir, session):
    filepath = os.path.join(output_dir, f"{pdb_id}.cif")
    if os.path.exists(filepath):
        return "skipped"

    url = f"https://files.rcsb.org/download/{pdb_id}.cif.gz"
    try:
        res = session.get(url, timeout=30)
        if res.status_code == 200:
            with open(filepath, "wb") as f:
                f.write(gzip.decompress(res.content))
            return "downloaded"
        return f"http_{res.status_code}"
    except Exception as e:
        return f"error: {e}"

def download_cifs_fast(pdb_ids, output_dir="raw_cifs", max_downloads=1000, max_workers=8):
    os.makedirs(output_dir, exist_ok=True)
    if max_downloads is not None:
        pdb_ids = pdb_ids[:max_downloads]

    total = len(pdb_ids)
    print(f"\nStarting parallel download of {total} structures ({max_workers} threads)...")

    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(pool_connections=max_workers, pool_maxsize=max_workers)
    session.mount("https://", adapter)

    done, downloaded, skipped, failed = 0, 0, 0, 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_download_one, pid, output_dir, session): pid for pid in pdb_ids}
        for future in as_completed(futures):
            status = future.result()
            done += 1
            if status == "downloaded":
                downloaded += 1
            elif status == "skipped":
                skipped += 1
            else:
                failed += 1

            if done % 100 == 0 or done == total:
                print(f"Progress: {done}/{total} | Downloaded: {downloaded} | Skipped: {skipped} | Failed: {failed}")

pdb_list = fetch_protein_na_pdb_ids()
download_cifs_fast(pdb_list, max_downloads=1000, max_workers=8)
print("\nProtein–Nucleic Acid dataset download complete.")

Fetching Protein–Nucleic Acid PDB IDs from RCSB API...
Found 5000 protein–nucleic acid complexes so far...
Found 10000 protein–nucleic acid complexes so far...
Found 15000 protein–nucleic acid complexes so far...
Found 17512 protein–nucleic acid complexes so far...

Starting parallel download of 1000 structures (8 threads)...
Progress: 100/1000 | Downloaded: 100 | Skipped: 0 | Failed: 0
Progress: 200/1000 | Downloaded: 200 | Skipped: 0 | Failed: 0
Progress: 300/1000 | Downloaded: 300 | Skipped: 0 | Failed: 0
Progress: 400/1000 | Downloaded: 400 | Skipped: 0 | Failed: 0
Progress: 500/1000 | Downloaded: 500 | Skipped: 0 | Failed: 0
Progress: 600/1000 | Downloaded: 600 | Skipped: 0 | Failed: 0
Progress: 700/1000 | Downloaded: 700 | Skipped: 0 | Failed: 0
Progress: 800/1000 | Downloaded: 800 | Skipped: 0 | Failed: 0
Progress: 900/1000 | Downloaded: 900 | Skipped: 0 | Failed: 0
Progress: 1000/1000 | Downloaded: 1000 | Skipped: 0 | Failed: 0

Protein–Nucleic Acid dataset download complete.


In [7]:
# @title 3. Clean Data & Generate Native Graphs (Multiprocessing + GPU cdist) { display-mode: "form" }
import os
import time
import warnings
import numpy as np
import torch
from concurrent.futures import ProcessPoolExecutor, as_completed
from Bio import BiopythonWarning
from Bio.PDB import MMCIFParser
from Bio.PDB.DSSP import DSSP
from torch_geometric.data import Data

warnings.simplefilter('ignore', UserWarning)
warnings.simplefilter('ignore', BiopythonWarning)

MAX_FILE_MB = 5.0   # Skips ribosomes / giant cryo-EM assemblies
MAX_NODES = 3000    # Standard docking complex size cap

# Precompute one-hot vectors once (avoids repeated np.eye/np.zeros allocations in loops)
AA_LIST = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
           'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']
NA_LIST = ['A', 'U', 'G', 'C', 'DA', 'DT', 'DG', 'DC']
SS_LIST = ['H', 'B', 'E', 'G', 'I', 'T', 'S', '-']

AA_OH = {aa: np.eye(20, dtype=np.float32)[i].tolist() for i, aa in enumerate(AA_LIST)}
NA_OH = {na: np.eye(8, dtype=np.float32)[i].tolist() for i, na in enumerate(NA_LIST)}
SS_OH = {ss: np.eye(8, dtype=np.float32)[i].tolist() for i, ss in enumerate(SS_LIST)}
ZERO_20 = [0.0] * 20
ZERO_8 = [0.0] * 8

def _parse_cif_worker(cif_path):
    """CPU worker: filters oversized files, parses mmCIF + DSSP, and returns raw numpy arrays."""
    warnings.simplefilter('ignore', UserWarning)
    warnings.simplefilter('ignore', BiopythonWarning)

    # 1. Skip oversized files immediately before parsing
    if os.path.getsize(cif_path) > MAX_FILE_MB * 1024 * 1024:
        return (os.path.basename(cif_path), "too_large_file", None, None)

    parser = MMCIFParser(QUIET=True)
    try:
        structure = parser.get_structure('complex', cif_path)
        model = structure[0]
    except Exception as e:
        return (os.path.basename(cif_path), f"error: {e}", None, None)

    # 2. Quick residue count BEFORE running DSSP
    valid_residues = []
    for chain in model:
        chain_id = chain.id
        for residue in chain:
            res_name = residue.resname.strip().upper()
            if 'CA' in residue and res_name in AA_OH:
                valid_residues.append((True, chain_id, residue, res_name, residue['CA'].get_coord()))
            elif 'P' in residue and res_name in NA_OH:
                valid_residues.append((False, chain_id, residue, res_name, residue['P'].get_coord()))

    if len(valid_residues) < 10:
        return (os.path.basename(cif_path), "too_small", None, None)
    if len(valid_residues) > MAX_NODES:
        return (os.path.basename(cif_path), "too_large_nodes", None, None)

    # 3. Run DSSP only on valid-sized complexes
    try:
        dssp = DSSP(model, cif_path, dssp='dssp')
    except Exception:
        dssp = {}

    coords, node_features = [], []
    for is_prot, chain_id, residue, res_name, coord in valid_residues:
        if is_prot:
            dssp_key = (chain_id, residue.id)
            if dssp_key in dssp:
                ss = dssp[dssp_key][2]
                sasa = float(dssp[dssp_key][3]) / 200.0
            else:
                ss = '-'
                sasa = 0.0
            feature_vector = [1.0, 0.0] + AA_OH[res_name] + ZERO_8 + SS_OH.get(ss, SS_OH['-']) + [sasa]
        else:
            feature_vector = [0.0, 1.0] + ZERO_20 + NA_OH[res_name] + ZERO_8 + [1.0]

        coords.append(coord)
        node_features.append(feature_vector)

    return (
        os.path.basename(cif_path),
        "ok",
        np.asarray(coords, dtype=np.float32),
        np.asarray(node_features, dtype=np.float32)
    )

def build_graph_gpu(coords_np, features_np, device):
    """Computes pairwise distances and edges on CUDA T4 GPU using torch.cdist."""
    pos_cpu = torch.from_numpy(coords_np)
    x_cpu = torch.from_numpy(features_np)

    try:
        pos_dev = pos_cpu.to(device)
        dist_mat = torch.cdist(pos_dev, pos_dev)
        mask = (dist_mat < 10.0) & (dist_mat > 0.1)
        edge_index = mask.nonzero(as_tuple=False).t().contiguous().cpu()
        del pos_dev, dist_mat, mask
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        dist_mat = torch.cdist(pos_cpu, pos_cpu)
        mask = (dist_mat < 10.0) & (dist_mat > 0.1)
        edge_index = mask.nonzero(as_tuple=False).t().contiguous()

    if edge_index.numel() == 0:
        return None

    return Data(
        x=x_cpu,
        edge_index=edge_index,
        pos=pos_cpu,
        y=torch.tensor([1.0], dtype=torch.float)
    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cif_files = [os.path.join("raw_cifs", f) for f in os.listdir("raw_cifs") if f.endswith('.cif')]
num_workers = os.cpu_count() or 2
t0 = time.time()

print(f"Converting {len(cif_files)} CIF files using {num_workers} CPU workers + {device} edge construction (max {MAX_FILE_MB} MB / {MAX_NODES} nodes)...")

native_graphs = []
skipped_large, skipped_other = 0, 0

with ProcessPoolExecutor(max_workers=num_workers) as executor:
    futures = [executor.submit(_parse_cif_worker, path) for path in cif_files]
    for idx, future in enumerate(as_completed(futures), 1):
        filename, status, coords_np, features_np = future.result()
        if status == "ok":
            g = build_graph_gpu(coords_np, features_np, device)
            if g is not None:
                native_graphs.append(g)
            else:
                skipped_other += 1
        elif status in ("too_large_file", "too_large_nodes"):
            skipped_large += 1
        else:
            skipped_other += 1
            if status.startswith("error:"):
                print(f"Skipping {filename} ({status})")

        if idx % 50 == 0 or idx == len(cif_files):
            elapsed = time.time() - t0
            print(f"Progress: {idx}/{len(cif_files)} | Valid: {len(native_graphs)} | Skipped (large): {skipped_large} | Skipped (other): {skipped_other} | Elapsed: {elapsed:.1f}s")

if device.type == 'cuda':
    torch.cuda.empty_cache()

print(f"\nDone in {time.time() - t0:.1f}s. Successfully created {len(native_graphs)} native graphs.")

Converting 1000 CIF files using 2 CPU workers + cuda edge construction (max 5.0 MB / 3000 nodes)...
Progress: 50/1000 | Valid: 38 | Skipped (large): 12 | Skipped (other): 0 | Elapsed: 130.3s
Progress: 100/1000 | Valid: 72 | Skipped (large): 28 | Skipped (other): 0 | Elapsed: 255.2s
Progress: 150/1000 | Valid: 107 | Skipped (large): 43 | Skipped (other): 0 | Elapsed: 325.4s
Progress: 200/1000 | Valid: 139 | Skipped (large): 61 | Skipped (other): 0 | Elapsed: 391.3s
Progress: 250/1000 | Valid: 180 | Skipped (large): 70 | Skipped (other): 0 | Elapsed: 490.1s
Progress: 300/1000 | Valid: 216 | Skipped (large): 84 | Skipped (other): 0 | Elapsed: 550.2s
Progress: 350/1000 | Valid: 253 | Skipped (large): 97 | Skipped (other): 0 | Elapsed: 632.5s
Progress: 400/1000 | Valid: 284 | Skipped (large): 116 | Skipped (other): 0 | Elapsed: 685.5s
Progress: 450/1000 | Valid: 320 | Skipped (large): 130 | Skipped (other): 0 | Elapsed: 743.3s
Progress: 500/1000 | Valid: 355 | Skipped (large): 145 | Skipped

In [8]:
# @title 4. Generate Decoys & Split Dataset (GPU-Accelerated) { display-mode: "form" }
import random
import torch
from torch_geometric.data import Data

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def random_rotation_matrix(device):
    """Generates a uniform random 3D rotation matrix (SO(3)) directly on GPU/CPU via unit quaternions."""
    q = torch.randn(4, device=device)
    q = q / q.norm()
    w, x, y, z = q[0], q[1], q[2], q[3]
    return torch.stack([
        torch.stack([1 - 2*(y*y + z*z), 2*(x*y - z*w),     2*(x*z + y*w)]),
        torch.stack([2*(x*y + z*w),     1 - 2*(x*x + z*z), 2*(y*z - x*w)]),
        torch.stack([2*(x*z - y*w),     2*(y*z + x*w),     1 - 2*(x*x + y*y)])
    ])

def _compute_decoy_on_device(pos_cpu, na_indices_cpu, translation_std, target_device):
    pos = pos_cpu.to(target_device).clone()
    na_indices = na_indices_cpu.to(target_device)

    na_coords = pos[na_indices]
    com = na_coords.mean(dim=0, keepdim=True)
    rot_matrix = random_rotation_matrix(target_device)
    translation = torch.randn(1, 3, device=target_device) * translation_std

    pos[na_indices] = (na_coords - com) @ rot_matrix.T + com + translation

    dist_mat = torch.cdist(pos, pos)
    mask = (dist_mat < 10.0) & (dist_mat > 0.1)
    edge_index = mask.nonzero(as_tuple=False).t().contiguous()

    return pos.cpu(), edge_index.cpu()

def generate_decoy_graph(native_graph, device, translation_std=15.0):
    na_indices = torch.where(native_graph.x[:, 1] == 1.0)[0]
    if len(na_indices) == 0:
        return None

    try:
        decoy_pos, edge_index = _compute_decoy_on_device(
            native_graph.pos, na_indices, translation_std, device
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        decoy_pos, edge_index = _compute_decoy_on_device(
            native_graph.pos, na_indices, translation_std, torch.device('cpu')
        )

    if edge_index.numel() == 0:
        return None

    return Data(
        x=native_graph.x,
        edge_index=edge_index,
        pos=decoy_pos,
        y=torch.tensor([0.0], dtype=torch.float)
    )

print(f"Generating decoys on {device} for {len(native_graphs)} native graphs...")

# Keep each native complex paired with its decoy before splitting to prevent train/val leakage
pairs = []
for idx, nat_g in enumerate(native_graphs, 1):
    decoy_g = generate_decoy_graph(nat_g, device)
    pairs.append((nat_g, decoy_g))
    if idx % 200 == 0 or idx == len(native_graphs):
        print(f"Progress: {idx}/{len(native_graphs)} complexes processed...")

if device.type == 'cuda':
    torch.cuda.empty_cache()

random.shuffle(pairs)
split = int(len(pairs) * 0.8)

train_dataset = [g for pair in pairs[:split] for g in pair if g is not None]
val_dataset = [g for pair in pairs[split:] for g in pair if g is not None]

random.shuffle(train_dataset)
random.shuffle(val_dataset)

print(f"\nDataset ready. Train: {len(train_dataset)} graphs | Validation: {len(val_dataset)} graphs")

Generating decoys on cuda for 703 native graphs...
Progress: 200/703 complexes processed...
Progress: 400/703 complexes processed...
Progress: 600/703 complexes processed...
Progress: 703/703 complexes processed...

Dataset ready. Train: 1124 graphs | Validation: 281 graphs


In [9]:
# @title 5. Load EGNN Architecture (Fast Native Scatter) { display-mode: "form" }
import torch
import torch.nn as nn
from torch_geometric.nn import global_mean_pool
from torch_geometric.utils import scatter

class EGNNLayer(nn.Module):
    def __init__(self, node_dim):
        super().__init__()
        self.node_mlp = nn.Sequential(
            nn.Linear(node_dim * 2 + 1, node_dim),
            nn.SiLU(),
            nn.Linear(node_dim, node_dim)
        )
        self.coord_mlp = nn.Sequential(
            nn.Linear(node_dim, node_dim),
            nn.SiLU(),
            nn.Linear(node_dim, 1)
        )

    def forward(self, x, pos, edge_index):
        src, dst = edge_index[0], edge_index[1]

        # Compute coordinate differences once and reuse for both radial distance and position update
        coord_diff = pos[dst] - pos[src]
        radial = (coord_diff * coord_diff).sum(dim=-1, keepdim=True)

        # Edge messages and equivariant coordinate displacements
        m_ij = self.node_mlp(torch.cat([x[dst], x[src], radial], dim=-1))
        pos_update = self.coord_mlp(m_ij) * coord_diff

        # Fast fused scatter-mean directly onto destination nodes
        num_nodes = x.size(0)
        agg_m = scatter(m_ij, dst, dim=0, dim_size=num_nodes, reduce='mean')
        agg_pos = scatter(pos_update, dst, dim=0, dim_size=num_nodes, reduce='mean')

        return x + agg_m, pos + agg_pos

class ComplexScoringModel(nn.Module):
    def __init__(self, in_node_dim=39, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Linear(in_node_dim, hidden_dim)
        self.egnn1 = EGNNLayer(hidden_dim)
        self.egnn2 = EGNNLayer(hidden_dim)
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.SiLU(),
            nn.Linear(32, 1)
        )

    def forward(self, data):
        x = self.embedding(data.x)
        x, pos = self.egnn1(x, data.pos, data.edge_index)
        x, pos = self.egnn2(x, pos, data.edge_index)
        graph_embedding = global_mean_pool(x, data.batch)
        return self.readout(graph_embedding)

print("Optimized EGNN Architecture Loaded.")

Optimized EGNN Architecture Loaded.


In [10]:
# @title 6. Train the Docking Model (T4 FP16 Mixed Precision + Validation) { display-mode: "form" }
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.loader import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
print(f"Executing on: {device} | FP16 Mixed Precision: {use_amp}")

batch_size = 16  # Larger batch size saturates T4 Tensor Cores (reduce to 8 if OOM)
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=2, pin_memory=use_amp, persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=2, pin_memory=use_amp, persistent_workers=True
)

model = ComplexScoringModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

epochs = 100
best_val_loss = float('inf')
best_model_state = None

for epoch in range(1, epochs + 1):
    # --- Training ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch in train_loader:
        batch = batch.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            out = model(batch).view(-1)
            loss = criterion(out, batch.y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        n = batch.num_graphs
        train_loss += loss.item() * n
        preds = (out.detach() > 0.0).float()  # Equivalent to sigmoid(out) > 0.5
        train_correct += (preds == batch.y).sum().item()
        train_total += n

    # --- Validation ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                out = model(batch).view(-1)
                loss = criterion(out, batch.y)

            n = batch.num_graphs
            val_loss += loss.item() * n
            preds = (out > 0.0).float()
            val_correct += (preds == batch.y).sum().item()
            val_total += n

    avg_train_loss = train_loss / train_total
    train_acc = train_correct / train_total
    avg_val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

# Restore best validation weights so Cell 7 saves the best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print(f"\nTraining Complete. Best Validation Loss: {best_val_loss:.4f}")

Executing on: cuda | FP16 Mixed Precision: True
Epoch 001 | Train Loss: 0.6994 | Train Acc: 0.4982 | Val Loss: 0.6932 | Val Acc: 0.4982
Epoch 002 | Train Loss: 0.6963 | Train Acc: 0.5089 | Val Loss: 0.6924 | Val Acc: 0.5338
Epoch 003 | Train Loss: 0.6969 | Train Acc: 0.5044 | Val Loss: 0.6929 | Val Acc: 0.4982
Epoch 004 | Train Loss: 0.6936 | Train Acc: 0.5178 | Val Loss: 0.6926 | Val Acc: 0.5018
Epoch 005 | Train Loss: 0.6932 | Train Acc: 0.5080 | Val Loss: 0.6920 | Val Acc: 0.5302
Epoch 006 | Train Loss: 0.6939 | Train Acc: 0.5098 | Val Loss: 0.6926 | Val Acc: 0.5018
Epoch 007 | Train Loss: 0.6929 | Train Acc: 0.4973 | Val Loss: 0.6918 | Val Acc: 0.5409
Epoch 008 | Train Loss: 0.6928 | Train Acc: 0.5187 | Val Loss: 0.6923 | Val Acc: 0.4982
Epoch 009 | Train Loss: 0.6928 | Train Acc: 0.5036 | Val Loss: 0.6920 | Val Acc: 0.4982
Epoch 010 | Train Loss: 0.6923 | Train Acc: 0.5231 | Val Loss: 0.6917 | Val Acc: 0.5480
Epoch 011 | Train Loss: 0.6852 | Train Acc: 0.5569 | Val Loss: 0.6588 | 

In [11]:
# @title 7. Save and Download Weights { display-mode: "form" }
import torch
from google.colab import files

save_path = "egnn_docking_scorer.pth"
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")
files.download(save_path)

Model saved to egnn_docking_scorer.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# @title 🎯🧠 Interactive AI Scoring Tool (Upload .pth & HDOCKlite Models) { display-mode: "form" }

!pip install biopython scipy requests torch_geometric -q
!apt-get update -qq
!apt-get install dssp -y -qq

import os
import re
import warnings
from Bio import BiopythonWarning
warnings.simplefilter('ignore', UserWarning)
warnings.simplefilter('ignore', BiopythonWarning)

from google.colab import files
from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP
import torch
import torch.nn as nn
from torch_geometric.data import Batch, Data
from torch_geometric.nn import MessagePassing, global_mean_pool
from scipy.spatial import distance_matrix
import numpy as np

# --- 1. Define the Core Architecture ---
class EGNNLayer(MessagePassing):
    def __init__(self, node_dim):
        super().__init__(aggr='mean')
        self.node_mlp = nn.Sequential(nn.Linear(node_dim * 2 + 1, node_dim), nn.SiLU(), nn.Linear(node_dim, node_dim))
        self.coord_mlp = nn.Sequential(nn.Linear(node_dim, node_dim), nn.SiLU(), nn.Linear(node_dim, 1))

    def forward(self, x, pos, edge_index):
        row, col = edge_index
        radial = torch.sum((pos[row] - pos[col]) ** 2, dim=1).unsqueeze(1)
        return self.propagate(edge_index, x=x, pos=pos, radial=radial)

    def message(self, x_i, x_j, radial, pos_i, pos_j):
        m_ij = self.node_mlp(torch.cat([x_i, x_j, radial], dim=-1))
        pos_update = self.coord_mlp(m_ij) * (pos_i - pos_j)
        return m_ij, pos_update

    def aggregate(self, inputs, index, ptr=None, dim_size=None):
        m_ij, pos_update = inputs
        return super().aggregate(m_ij, index, ptr=ptr, dim_size=dim_size), super().aggregate(pos_update, index, ptr=ptr, dim_size=dim_size)

    def update(self, aggr_out, x, pos):
        agg_m, agg_pos = aggr_out
        return x + agg_m, pos + agg_pos

class ComplexScoringModel(nn.Module):
    def __init__(self, in_node_dim=39, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Linear(in_node_dim, hidden_dim)
        self.egnn1 = EGNNLayer(hidden_dim)
        self.egnn2 = EGNNLayer(hidden_dim)
        self.readout = nn.Sequential(nn.Linear(hidden_dim, 32), nn.SiLU(), nn.Linear(32, 1))

    def forward(self, data):
        x = self.embedding(data.x)
        x, pos = self.egnn1(x, data.pos, data.edge_index)
        x, pos = self.egnn2(x, pos, data.edge_index)
        graph_embedding = global_mean_pool(x, data.batch)
        return self.readout(graph_embedding)

# --- 2. Setup Biophysics Mappings ---
AA_MAP = {aa: i for i, aa in enumerate(['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL'])}
NA_MAP = {na: i for i, na in enumerate(['A', 'U', 'G', 'C', 'DA', 'DT', 'DG', 'DC'])}
SS_MAP = {'H': 0, 'B': 1, 'E': 2, 'G': 3, 'I': 4, 'T': 5, 'S': 6, '-': 7}

# --- 3. Upload Model Weights (.pth) & PDB Files (model_x.pdb) ---
work_dir = '/content/uploaded_hdock'
os.makedirs(work_dir, exist_ok=True)

print("📤 Step 1: Upload your trained model weights (.pth file)...")
uploaded_pth = files.upload()
pth_files = [f for f in uploaded_pth.keys() if f.endswith('.pth')]
if not pth_files:
    raise FileNotFoundError("No .pth file was uploaded.")

model_filename = pth_files[0]
model_path = os.path.join(work_dir, model_filename)
with open(model_path, 'wb') as f:
    f.write(uploaded_pth[model_filename])

print("\n📤 Step 2: Upload your HDOCKlite PDB files (model_1.pdb, model_2.pdb, etc.)...")
uploaded_pdbs = files.upload()

pdb_files = []
for fname, content in uploaded_pdbs.items():
    if fname.endswith('.pdb'):
        fpath = os.path.join(work_dir, fname)
        with open(fpath, 'wb') as f:
            f.write(content)
        pdb_files.append(fpath)

# Sort naturally (e.g., model_1.pdb, model_2.pdb, ..., model_10.pdb)
pdb_files.sort(key=lambda p: [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', os.path.basename(p))])

if not pdb_files:
    print("No .pdb files were uploaded. Exiting.")
else:
    print(f"\n✅ Loaded weights: {model_path}")
    print(f"⏳ Scoring {len(pdb_files)} uploaded model(s)...\n")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    scorer = ComplexScoringModel(in_node_dim=39, hidden_dim=64).to(device)
    scorer.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    scorer.eval()

    parser = PDBParser(QUIET=True)
    results = []

    for pdb_path in pdb_files:
        pdb_filename = os.path.basename(pdb_path)
        structure = parser.get_structure('hdock_pose', pdb_path)
        model_struct = structure[0]

        try:
            dssp = DSSP(model_struct, pdb_path, dssp='dssp')
        except Exception:
            dssp = {}

        coords, node_features = [], []

        for chain in model_struct:
            for residue in chain:
                res_name = residue.resname.strip().upper()
                if 'CA' in residue and res_name in AA_MAP:
                    atom = residue['CA']
                    is_prot = 1.0
                    id_vec = np.eye(20)[AA_MAP[res_name]].tolist()
                    dssp_key = (chain.id, residue.id)
                    ss = dssp[dssp_key][2] if dssp_key in dssp else '-'
                    sasa = float(dssp[dssp_key][3]) / 200.0 if dssp_key in dssp else 0.0
                    ss_vec = np.eye(8)[SS_MAP.get(ss, 7)].tolist()
                    na_vec = np.zeros(8).tolist()
                elif 'P' in residue and res_name in NA_MAP:
                    atom = residue['P']
                    is_prot = 0.0
                    id_vec = np.zeros(20).tolist()
                    ss_vec = np.zeros(8).tolist()
                    sasa = 1.0
                    na_vec = np.eye(8)[NA_MAP.get(res_name, 0)].tolist()
                else:
                    continue

                coords.append(atom.get_coord())
                feature_vector = [is_prot, 1.0 - is_prot] + id_vec + na_vec + ss_vec + [sasa]
                node_features.append(feature_vector)

        if len(coords) < 10:
            print(f"❌ {pdb_filename}: Error - Could not extract enough valid atoms.")
            continue

        coords = np.array(coords)
        dist_mat = distance_matrix(coords, coords)
        edges = np.where((dist_mat < 10.0) & (dist_mat > 0.1))

        if len(edges[0]) == 0:
            print(f"❌ {pdb_filename}: Error - Graph has no connecting edges.")
            continue

        graph = Data(
            x=torch.tensor(node_features, dtype=torch.float),
            edge_index=torch.tensor(np.vstack(edges), dtype=torch.long),
            pos=torch.tensor(coords, dtype=torch.float)
        )

        batch = Batch.from_data_list([graph]).to(device)
        with torch.no_grad():
            raw_logit = scorer(batch).squeeze()
            if raw_logit.dim() == 0:
                raw_logit = raw_logit.unsqueeze(0)
            probability = torch.sigmoid(raw_logit).item()

        results.append((pdb_filename, probability))

    # Sort results by highest probability and print rankings
    results.sort(key=lambda x: x[1], reverse=True)

    print("\n--- FINAL AI RANKINGS ---")
    for rank, (filename, prob) in enumerate(results, 1):
        print(f"Rank {rank}: {filename} -> Native Probability: {prob * 100:.2f}%")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
📤 Step 1: Upload your trained model weights (.pth file)...


Saving egnn_docking_scorer.pth to egnn_docking_scorer (1).pth

📤 Step 2: Upload your HDOCKlite PDB files (model_1.pdb, model_2.pdb, etc.)...


Saving model_1.pdb to model_1.pdb
Saving model_2.pdb to model_2.pdb
Saving model_3.pdb to model_3.pdb
Saving model_4.pdb to model_4.pdb
Saving model_5.pdb to model_5.pdb
Saving model_6.pdb to model_6.pdb
Saving model_7.pdb to model_7.pdb
Saving model_8.pdb to model_8.pdb
Saving model_9.pdb to model_9.pdb
Saving model_10.pdb to model_10.pdb
Saving model_11.pdb to model_11.pdb
Saving model_12.pdb to model_12.pdb
Saving model_13.pdb to model_13.pdb
Saving model_14.pdb to model_14.pdb
Saving model_15.pdb to model_15.pdb
Saving model_16.pdb to model_16.pdb
Saving model_17.pdb to model_17.pdb
Saving model_18.pdb to model_18.pdb
Saving model_19.pdb to model_19.pdb
Saving model_20.pdb to model_20.pdb
Saving model_21.pdb to model_21.pdb
Saving model_22.pdb to model_22.pdb
Saving model_23.pdb to model_23.pdb
Saving model_24.pdb to model_24.pdb
Saving model_25.pdb to model_25.pdb
Saving model_26.pdb to model_26.pdb
Saving model_27.pdb to model_27.pdb
Saving model_28.pdb to model_28.pdb
Saving mod

In [14]:
# @title 8. Benchmark EGNN Re-Ranking vs. HDOCKlite (DockQ) { display-mode: "form" }
native_pdb_path = "/content/drive/MyDrive/Training_Set_Docking/REV-RNA.pdb" # @param {type:"string"}
dockq_acceptable_threshold = 0.23 # @param {type:"number"}

!pip install -q DockQ

import os
import re
import shutil
import subprocess
import pandas as pd
from scipy.stats import spearmanr

DOCKQ_BIN = shutil.which("DockQ")
if DOCKQ_BIN is None:
    raise RuntimeError("DockQ installation failed or is not on PATH.")

_printed_dockq_err = False

def compute_dockq(model_pdb, native_pdb):
    """Runs DockQ CLI and parses the DockQ score (supports v1 & v2)."""
    global _printed_dockq_err
    cmd = [DOCKQ_BIN, model_pdb, native_pdb, "--short"]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    match = re.search(r"DockQ[:\s]+([0-9]*\.?[0-9]+)", proc.stdout)
    if match:
        return float(match.group(1))
    if not _printed_dockq_err:
        print(f"⚠️ DockQ CLI failed on {os.path.basename(model_pdb)}:")
        print("STDOUT:", proc.stdout[:300])
        print("STDERR:", proc.stderr[:300])
        _printed_dockq_err = True
    return 0.0

if 'results' not in globals() or not results:
    print("Run the Interactive AI Scoring Tool cell first to populate `results`.")
elif not os.path.exists(native_pdb_path):
    print(f"Native reference PDB not found at: {native_pdb_path}")
else:
    rows = []
    for egnn_rank, (fname, prob) in enumerate(results, 1):
        hdock_rank = int(re.search(r"model_(\d+)\.pdb", fname).group(1))
        pdb_path = os.path.join(work_dir, fname)
        dq = compute_dockq(pdb_path, native_pdb_path)
        rows.append({
            "Model": fname,
            "HDOCKlite_Rank": hdock_rank,
            "EGNN_Rank": egnn_rank,
            "EGNN_Prob": round(prob, 4),
            "DockQ": round(dq, 4),
            "Acceptable": dq >= dockq_acceptable_threshold
        })

    bench_df = pd.DataFrame(rows)
    bench_csv = os.path.join(work_dir, "benchmark_hdock_vs_egnn.csv")
    bench_df.to_csv(bench_csv, index=False)

    # Top-N Success Rate comparison
    print("=== BENCHMARK: HDOCKlite Default vs. EGNN Re-Ranking ===")
    for k in [1, 5, 10]:
        hdock_top_k = bench_df.nsmallest(k, "HDOCKlite_Rank")["DockQ"].max()
        egnn_top_k = bench_df.nsmallest(k, "EGNN_Rank")["DockQ"].max()
        print(
            f"Top-{k:02d} Best DockQ | "
            f"HDOCKlite: {hdock_top_k:.4f} ({'PASS' if hdock_top_k >= dockq_acceptable_threshold else 'FAIL'}) | "
            f"EGNN: {egnn_top_k:.4f} ({'PASS' if egnn_top_k >= dockq_acceptable_threshold else 'FAIL'})"
        )

    rho_egnn, p_egnn = spearmanr(bench_df["EGNN_Prob"], bench_df["DockQ"])
    rho_hdock, p_hdock = spearmanr(-bench_df["HDOCKlite_Rank"], bench_df["DockQ"])
    print(f"\nSpearman Correlation with DockQ:")
    print(f"  • HDOCKlite Default Rank: rho = {rho_hdock:.4f} (p = {p_hdock:.2e})")
    print(f"  • EGNN Native Probability: rho = {rho_egnn:.4f} (p = {p_egnn:.2e})")
    print(f"\nFull benchmark table saved to: {bench_csv}")
    display(bench_df.sort_values("EGNN_Rank").head(10))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.11.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.11.1 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.11.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.10.15 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda13x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires nump

,Model,HDOCKlite_Rank,EGNN_Rank,EGNN_Prob,DockQ,Acceptable
0,model_23.pdb,23,1,1.0,0.060,False
1,model_81.pdb,81,2,1.0,0.025,False
2,model_11.pdb,11,3,1.0,0.025,False
3,model_43.pdb,43,4,1.0,0.043,False
4,model_38.pdb,38,5,1.0,0.016,False
5,model_59.pdb,59,6,1.0,0.024,False
6,model_33.pdb,33,7,1.0,0.021,False
7,model_6.pdb,6,8,1.0,0.021,False
8,model_56.pdb,56,9,1.0,0.035,False
9,model_28.pdb,28,10,1.0,0.043,False


In [18]:
# @title 🧬 Visualize Rank 1 Complex & Interactions (py3Dmol) { display-mode: "form" }
interaction_cutoff = 4.5 # @param {type:"number"}
image_width = 1200 # @param {type:"integer"}
image_height = 800 # @param {type:"integer"}

!pip install py3Dmol -q

import os
import py3Dmol
from Bio.PDB import PDBParser, NeighborSearch
from IPython.display import display, HTML

if 'results' not in globals() or len(results) < 1:
    print("⚠️ No ranked models found. Please run the AI Scoring cell first.")
else:
    rank1_filename, rank1_prob = results[0]
    rank1_path = os.path.join(work_dir, rank1_filename)
    print(f"🥇 Rank 1 Model: {rank1_filename} | Native Probability: {rank1_prob * 100:.2f}%")

    # --- 1. Identify Protein–Nucleic Acid Interacting Residues ---
    AA_SET = {'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
              'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL'}
    NA_SET = {'A', 'U', 'G', 'C', 'DA', 'DT', 'DG', 'DC', 'ADE', 'URA', 'GUA', 'CYT', 'THY'}

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('rank1', rank1_path)
    model = structure[0]

    prot_atoms, na_atoms = [], []
    for chain in model:
        for res in chain:
            resname = res.resname.strip().upper()
            if resname in AA_SET:
                prot_atoms.extend(res.get_atoms())
            elif resname in NA_SET:
                na_atoms.extend(res.get_atoms())

    interacting_prot = {}
    interacting_na = {}

    if prot_atoms and na_atoms:
        ns_prot = NeighborSearch(prot_atoms)
        for atom in na_atoms:
            close_prot = ns_prot.search(atom.get_coord(), interaction_cutoff, level='R')
            if close_prot:
                na_res = atom.get_parent()
                na_key = (na_res.get_parent().id, na_res.id[1])
                if na_key not in interacting_na:
                    anchor = na_res['P'].get_coord() if 'P' in na_res else atom.get_coord()
                    interacting_na[na_key] = (na_res.resname.strip(), anchor)
                for p_res in close_prot:
                    p_key = (p_res.get_parent().id, p_res.id[1])
                    if p_key not in interacting_prot:
                        anchor = p_res['CA'].get_coord() if 'CA' in p_res else next(p_res.get_atoms()).get_coord()
                        interacting_prot[p_key] = (p_res.resname.strip(), anchor)

    print(f"🔗 Found {len(interacting_prot)} protein & {len(interacting_na)} nucleic acid residues within {interaction_cutoff} Å.")
    if interacting_prot:
        print("   Protein:", ", ".join(f"{v[0]}{k[1]}({k[0]})" for k, v in sorted(interacting_prot.items())))
    if interacting_na:
        print("   Nucleic Acid:", ", ".join(f"{v[0]}{k[1]}({k[0]})" for k, v in sorted(interacting_na.items())))

    # --- 2. Build py3Dmol Visualization ---
    with open(rank1_path, 'r') as f:
        pdb_data = f.read()

    view = py3Dmol.view(width=image_width, height=image_height)
    view.addModel(pdb_data, 'pdb')
    view.setBackgroundColor('white')
    view.setStyle({'cartoon': {'colorscheme': 'chain', 'opacity': 0.85}})

    # Highlight & label interacting protein residues (cyan sticks)
    for (chain_id, resi), (resname, coord) in interacting_prot.items():
        view.addStyle({'chain': chain_id, 'resi': resi}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.2}})
        view.addLabel(
            f"{resname}{resi}:{chain_id}",
            {'position': {'x': float(coord[0]), 'y': float(coord[1]), 'z': float(coord[2])},
             'backgroundColor': '#005f73', 'fontColor': 'white', 'fontSize': 11,
             'backgroundOpacity': 0.85, 'borderThickness': 0}
        )

    # Highlight & label interacting nucleic acid residues (orange sticks)
    for (chain_id, resi), (resname, coord) in interacting_na.items():
        view.addStyle({'chain': chain_id, 'resi': resi}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.2}})
        view.addLabel(
            f"{resname}{resi}:{chain_id}",
            {'position': {'x': float(coord[0]), 'y': float(coord[1]), 'z': float(coord[2])},
             'backgroundColor': '#ae2012', 'fontColor': 'white', 'fontSize': 11,
             'backgroundOpacity': 0.85, 'borderThickness': 0}
        )

    view.zoomTo()

    # --- 3. Display Viewer + Reliable High-Res PNG Download (via Colab Callback) ---
    import base64
    from google.colab import output, files

    png_filename = rank1_filename.replace('.pdb', '_interactions.png')
    png_save_path = os.path.join(work_dir, png_filename)

    def _save_and_download_png(data_uri):
        header, b64data = data_uri.split(',', 1)
        with open(png_save_path, 'wb') as f:
            f.write(base64.b64decode(b64data))
        print(f"✅ Saved high-res PNG to: {png_save_path}")
        files.download(png_save_path)

    output.register_callback('notebook.save_py3dmol_png', _save_and_download_png)

    # Generate HTML first so py3Dmol assigns view.uniqueid
    viewer_html = view._make_html()
    vid = str(view.uniqueid) if view.uniqueid is not None else "0"

    download_btn = f"""
    <div id="py3dmol_wrap_{vid}">
        {viewer_html}
    </div>
    <div style="margin-top: 10px;">
        <button onclick="captureAndDownload_{vid}()"
                style="padding: 10px 18px; font-size: 14px; font-weight: bold; color: white;
                       background-color: #1a73e8; border: none; border-radius: 6px; cursor: pointer;">
            📸 Save & Download High-Res PNG ({image_width}x{image_height})
        </button>
        <span id="dl_status_{vid}" style="margin-left: 10px; font-family: sans-serif; font-size: 13px;"></span>
    </div>
    <script>
        function captureAndDownload_{vid}() {{
            var status = document.getElementById('dl_status_{vid}');
            status.innerText = '⏳ Capturing...';
            try {{
                var v = window['viewer_{vid}'];
                var uri = null;
                if (v && typeof v.pngURI === 'function') {{
                    v.render();
                    uri = v.pngURI();
                }} else {{
                    var canvas = document.querySelector('#py3dmol_wrap_{vid} canvas');
                    if (canvas) uri = canvas.toDataURL('image/png');
                }}
                if (!uri) {{
                    status.innerText = '❌ Could not find 3Dmol viewer canvas.';
                    return;
                }}
                google.colab.kernel.invokeFunction('notebook.save_py3dmol_png', [uri], {{}})
                    .then(() => {{ status.innerText = '✅ Downloaded & saved to Drive!'; }})
                    .catch(err => {{ status.innerText = '❌ Callback error: ' + err; }});
            }} catch (e) {{
                status.innerText = '❌ Error: ' + e.message;
            }}
        }}
    </script>
    """
    display(HTML(download_btn))

🥇 Rank 1 Model: model_23.pdb | Native Probability: 100.00%
🔗 Found 17 protein & 15 nucleic acid residues within 4.5 Å.
   Protein: LEU22(A), GLN36(A), ASN40(A), ARG43(A), ARG44(A), TRP45(A), ARG46(A), GLU47(A), ARG48(A), GLN49(A), ARG50(A), GLN51(A), HIS53(A), SER54(A), ILE55(A), ARG58(A), ILE59(A)
   Nucleic Acid: G2( ), C3( ), U4( ), G5( ), G6( ), A7( ), C8( ), U9( ), C10( ), U20( ), A21( ), C22( ), U23( ), G25( ), A26( )


✅ Saved high-res PNG to: /content/uploaded_hdock/model_23_interactions.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
# @title 📊 Protein–Nucleic Acid Interaction Table (Rank 1 — CSV & High-Res PNG) { display-mode: "form" }
interaction_cutoff = 4.5 # @param {type:"number"}
show_all_atom_pairs = True # @param {type:"boolean"}

import os
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from google.colab import files
from Bio.PDB import PDBParser, NeighborSearch

if 'rank1_path' not in globals() or not os.path.exists(rank1_path):
    print("⚠️ Rank 1 PDB not found. Please run the Visualization cell first.")
else:
    AA_SET = {'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
              'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL'}
    NA_SET = {'A', 'U', 'G', 'C', 'DA', 'DT', 'DG', 'DC', 'ADE', 'URA', 'GUA', 'CYT', 'THY'}
    POS_AA = {'ARG', 'LYS', 'HIS'}
    PHOS_ATOMS = {'P', 'OP1', 'OP2', 'O1P', 'O2P'}

    def classify_interaction(p_res, p_atom, na_atom, dist):
        p_el = p_atom.element.upper() if p_atom.element else p_atom.get_name()[0]
        na_el = na_atom.element.upper() if na_atom.element else na_atom.get_name()[0]
        if p_res in POS_AA and p_el == 'N' and na_atom.get_name() in PHOS_ATOMS and dist <= 4.0:
            return "Electrostatic (Salt Bridge)"
        if p_el in {'N', 'O'} and na_el in {'N', 'O'} and dist <= 3.5:
            return "Hydrogen Bond / Polar"
        if p_el == 'C' and na_el == 'C' and dist <= 4.0:
            return "Hydrophobic / π-Stacking"
        return "Van der Waals / Contact"

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('rank1', rank1_path)
    model = structure[0]

    prot_atoms, na_atoms = [], []
    for chain in model:
        for res in chain:
            resname = res.resname.strip().upper()
            if resname in AA_SET:
                prot_atoms.extend(res.get_atoms())
            elif resname in NA_SET:
                na_atoms.extend(res.get_atoms())

    records = []
    if prot_atoms and na_atoms:
        ns_prot = NeighborSearch(prot_atoms)
        for na_atom in na_atoms:
            close_p_atoms = ns_prot.search(na_atom.get_coord(), interaction_cutoff, level='A')
            na_res = na_atom.get_parent()
            na_chain = na_res.get_parent().id
            for p_atom in close_p_atoms:
                p_res = p_atom.get_parent()
                p_chain = p_res.get_parent().id
                dist = float(p_atom - na_atom)
                records.append({
                    'Protein Chain': p_chain,
                    'Protein Residue': f"{p_res.resname.strip()}{p_res.id[1]}",
                    'Protein Atom': p_atom.get_name(),
                    'NA Chain': na_chain,
                    'NA Residue': f"{na_res.resname.strip()}{na_res.id[1]}",
                    'NA Atom': na_atom.get_name(),
                    'Distance (Å)': round(dist, 2),
                    'Interaction Type': classify_interaction(p_res.resname.strip().upper(), p_atom, na_atom, dist)
                })

    if not records:
        print(f"No protein–nucleic acid interactions found within {interaction_cutoff} Å.")
    else:
        df = pd.DataFrame(records).sort_values('Distance (Å)')
        if not show_all_atom_pairs:
            # Keep the closest atom contact per residue-residue pair
            df = df.drop_duplicates(subset=['Protein Chain', 'Protein Residue', 'NA Chain', 'NA Residue'])
        df = df.reset_index(drop=True)

        # --- Save CSV ---
        base_name = rank1_filename.replace('.pdb', '')
        csv_path = os.path.join(work_dir, f"{base_name}_interactions.csv")
        png_path = os.path.join(work_dir, f"{base_name}_interactions_table.png")
        df.to_csv(csv_path, index=False)

        # --- Render High-Resolution PNG Table (300 DPI) ---
        row_height = 0.35
        fig_height = max(2.5, (len(df) + 2) * row_height)
        fig, ax = plt.subplots(figsize=(12, fig_height), dpi=300)
        ax.axis('off')
        ax.set_title(
            f"Protein–Nucleic Acid Interactions | {rank1_filename} (Cutoff ≤ {interaction_cutoff} Å)",
            fontsize=13, fontweight='bold', pad=12, color='#1b263b'
        )

        table = ax.table(
            cellText=df.values,
            colLabels=df.columns,
            loc='center',
            cellLoc='center'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 1.4)

        for (row, col), cell in table.get_celld().items():
            cell.set_edgecolor('#d0d7de')
            if row == 0:
                cell.set_facecolor('#1b263b')
                cell.set_text_props(color='white', fontweight='bold')
            else:
                cell.set_facecolor('#f6f8fa' if row % 2 == 0 else 'white')

        plt.tight_layout()
        plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        print(f"✅ Found {len(df)} interaction(s). Saved to Google Drive:\n   • {csv_path}\n   • {png_path}")

        # --- Download Buttons ---
        btn_csv = widgets.Button(description='⬇️ Download CSV', button_style='info')
        btn_png = widgets.Button(description='⬇️ Download High-Res PNG', button_style='success')

        btn_csv.on_click(lambda b: files.download(csv_path))
        btn_png.on_click(lambda b: files.download(png_path))
        display(widgets.HBox([btn_csv, btn_png]))

✅ Found 730 interaction(s). Saved to Google Drive:
   • /content/uploaded_hdock/model_23_interactions.csv
   • /content/uploaded_hdock/model_23_interactions_table.png
